# 03 - POS Tagging & Chunking
**MIDTERM: Bigram tagger training & evaluation**
**FINAL: Chunking with grammar rules, NP extraction**

In [ ]:
# === IMPORTS ===
import nltk
from nltk.corpus import brown
from nltk.tag import UnigramTagger, BigramTagger, DefaultTagger
from nltk import RegexpParser

---
## PART A: POS TAGGING (Midterm)
---

### PATTERN 1: Get Tagged Sentences from Brown

In [ ]:
# Get tagged sentences with universal tagset
tagged_sents = brown.tagged_sents(tagset='universal')
print('Total sentences:', len(tagged_sents))
print('First sentence:', tagged_sents[0])

### PATTERN 2: Train/Test Split

In [ ]:
# Split 90% train, 10% test
size = int(len(tagged_sents) * 0.9)
train_sents = tagged_sents[:size]
test_sents = tagged_sents[size:]
print(f'Train: {len(train_sents)}, Test: {len(test_sents)}')

### PATTERN 3: Train Bigram Tagger with Backoff Chain

In [ ]:
# Best practice: Backoff chain
# Bigram -> Unigram -> Default

# Step 1: Default tagger (fallback for unknown words)
t0 = DefaultTagger('NOUN')  # Most common tag as default

# Step 2: Unigram tagger (backs off to default)
t1 = UnigramTagger(train_sents, backoff=t0)

# Step 3: Bigram tagger (backs off to unigram)
t2 = BigramTagger(train_sents, backoff=t1)

print('Tagger chain created!')

### PATTERN 4: Evaluate Tagger Accuracy

In [ ]:
# Evaluate on test set
accuracy = t2.accuracy(test_sents)
print(f'Bigram tagger accuracy: {accuracy:.4f}')
print(f'Bigram tagger accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Compare individual tagger accuracies
print(f'Default tagger: {t0.accuracy(test_sents):.4f}')
print(f'Unigram tagger: {t1.accuracy(test_sents):.4f}')
print(f'Bigram tagger:  {t2.accuracy(test_sents):.4f}')

### PATTERN 5: Tag New Sentences

In [ ]:
# Tag a new sentence
sentence = "The quick brown fox jumps over the lazy dog"
tokens = nltk.word_tokenize(sentence)

# Using our trained tagger
tagged = t2.tag(tokens)
print('Tagged sentence:', tagged)

In [ ]:
# Using NLTK's built-in POS tagger
tagged_builtin = nltk.pos_tag(tokens)
print('Built-in tagger:', tagged_builtin)

### COMPLETE EXAMPLE: Train & Evaluate Tagger

In [ ]:
def train_and_evaluate_tagger(train_ratio=0.9):
    """Train bigram tagger with backoff and evaluate"""
    # Get data
    tagged_sents = brown.tagged_sents(tagset='universal')
    
    # Split
    size = int(len(tagged_sents) * train_ratio)
    train_sents = tagged_sents[:size]
    test_sents = tagged_sents[size:]
    
    # Build backoff chain
    t0 = DefaultTagger('NOUN')
    t1 = UnigramTagger(train_sents, backoff=t0)
    t2 = BigramTagger(train_sents, backoff=t1)
    
    # Evaluate
    accuracy = t2.accuracy(test_sents)
    print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')
    
    return t2

# Usage
tagger = train_and_evaluate_tagger()

---
## PART B: CHUNKING (Final)
---

### PATTERN 6: Basic NP Chunking Grammar

In [ ]:
# Simple NP grammar: optional determiner, any adjectives, noun
grammar = "NP: {<DT>?<JJ>*<NN>}"

# Create parser
cp = nltk.RegexpParser(grammar)

In [ ]:
# Example tagged sentence
sentence = [("the", "DT"), ("little", "JJ"), ("yellow", "JJ"),
            ("dog", "NN"), ("barked", "VBD"), ("at", "IN"), 
            ("the", "DT"), ("cat", "NN")]

# Parse and print result
result = cp.parse(sentence)
print(result)

### PATTERN 7: Extended NP Grammar

In [ ]:
# More complete NP grammar
grammar = r"""
NP: {<DT|PP\$>?<JJ>*<NN>}   # Chunk: det/possessive + adj + noun
    {<NNP>+}                 # Chunk: proper nouns
"""

cp = nltk.RegexpParser(grammar)

sentence = [("Rapunzel", "NNP"), ("let", "VBD"), ("down", "RP"),
            ("her", "PP$"), ("long", "JJ"), ("golden", "JJ"), ("hair", "NN")]

print(cp.parse(sentence))

### PATTERN 8: Extract NP Chunks

In [ ]:
# Extract all NP chunks from parsed result
grammar = "NP: {<DT>?<JJ>*<NN>}"
cp = nltk.RegexpParser(grammar)

sentence = [("the", "DT"), ("little", "JJ"), ("yellow", "JJ"),
            ("dog", "NN"), ("barked", "VBD"), ("at", "IN"), 
            ("the", "DT"), ("cat", "NN")]

tree = cp.parse(sentence)

# Extract NP subtrees
for subtree in tree.subtrees():
    if subtree.label() == 'NP':
        print('NP found:', subtree)

### PATTERN 9: Complete Workflow - Tokenize -> Tag -> Chunk

In [ ]:
def extract_noun_phrases(text):
    """Complete pipeline: text -> tokenize -> tag -> chunk -> extract NPs"""
    # Step 1: Tokenize
    tokens = nltk.word_tokenize(text)
    
    # Step 2: POS Tag
    tagged = nltk.pos_tag(tokens)
    
    # Step 3: Define grammar and parse
    grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
    cp = nltk.RegexpParser(grammar)
    tree = cp.parse(tagged)
    
    # Step 4: Extract NPs
    noun_phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == 'NP':
            np = ' '.join(word for word, tag in subtree.leaves())
            noun_phrases.append(np)
    
    return noun_phrases

# Usage
text = "The quick brown fox jumps over the lazy dog in the park."
nps = extract_noun_phrases(text)
print('Noun phrases:', nps)

### PATTERN 10: IE Preprocess (from lectures)

In [ ]:
def ie_preprocess(document):
    """Preprocess text for information extraction"""
    sentences = nltk.sent_tokenize(document)
    sentences = [nltk.word_tokenize(sent) for sent in sentences]
    sentences = [nltk.pos_tag(sent) for sent in sentences]
    return sentences

# Usage
text = "The cat sat on the mat. The dog ran in the park."
processed = ie_preprocess(text)
for sent in processed:
    print(sent)

---
## GRAMMAR PATTERNS QUICK REFERENCE

In [ ]:
# Common grammar patterns:

# Basic NP: det + adj* + noun
# grammar = "NP: {<DT>?<JJ>*<NN>}"

# NP with proper nouns
# grammar = "NP: {<DT>?<JJ>*<NN>}\n    {<NNP>+}"

# VP: verb + NP
# grammar = "VP: {<VB.*><NP>}"

# PP: preposition + NP  
# grammar = "PP: {<IN><NP>}"

# Chinking (exclude patterns)
# grammar = r"""NP: {<.*>+}         # Chunk everything
#                   }<VBD|IN>+{     # Chink VBD and IN"""

print("Grammar patterns loaded!")

---
## POS TAG REFERENCE

In [ ]:
# Universal tagset:
# NOUN - noun
# VERB - verb  
# ADJ  - adjective
# ADV  - adverb
# DET  - determiner
# ADP  - adposition (preposition)
# PRON - pronoun
# CONJ - conjunction
# PRT  - particle
# NUM  - number
# .    - punctuation
# X    - other

# Penn Treebank (default nltk.pos_tag):
# NN   - noun, singular
# NNS  - noun, plural
# NNP  - proper noun, singular
# VB   - verb, base form
# VBD  - verb, past tense
# VBG  - verb, gerund
# JJ   - adjective
# RB   - adverb
# DT   - determiner
# IN   - preposition
# PP$  - possessive pronoun

print("Tag reference loaded!")